In [ ]:
#install deps
!pip install transformers datasets torch accelerate scikit-learn optimum onnxruntime onnx -q

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#load data
en=load_dataset('xnli','en',split='train')
hi=load_dataset('xnli','hi',split='validation')

#take 20%
sub_en=en.shuffle(seed=42).select(range(int(len(en)*0.2)))
print(len(sub_en), len(hi))

In [ ]:
mdl_name="xlm-roberta-base"
tok=AutoTokenizer.from_pretrained(mdl_name)

def tok_fn(x):
    return tok(x['premise'],x['hypothesis'],truncation=True,max_length=128)

tok_en=sub_en.map(tok_fn,batched=True,remove_columns=['premise','hypothesis'])
tok_hi=hi.map(tok_fn,batched=True,remove_columns=['premise','hypothesis'])

In [ ]:
lbls={0:"entailment",1:"neutral",2:"contradiction"}
id2lbl={v:k for k,v in lbls.items()}

model=AutoModelForSequenceClassification.from_pretrained(mdl_name,num_labels=3,id2label=lbls,label2id=id2lbl)
dc=DataCollatorWithPadding(tokenizer=tok)

In [ ]:
def acc(p):
    l,lb=p
    preds=np.argmax(l,axis=-1)
    return {"acc":accuracy_score(lb,preds)}

split=tok_en.train_test_split(test_size=0.1,seed=42)

args=TrainingArguments(
    output_dir="./res",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available()
)

tr=Trainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    tokenizer=tok,
    data_collator=dc,
    compute_metrics=acc
)

tr.train()

In [ ]:
#eval on hindi
preds=tr.predict(tok_hi)
p_lbls=np.argmax(preds.predictions,axis=-1)
ac=accuracy_score(preds.label_ids,p_lbls)
f1=f1_score(preds.label_ids,p_lbls,average='weighted')
print("hindi acc:",ac)
print("hindi f1:",f1)

In [ ]:
#save
tr.save_model("my_model")
tok.save_pretrained("my_model")

In [ ]:
#mbert baseline comparison
m_tok=AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
def m_tok_fn(x):
    return m_tok(x['premise'],x['hypothesis'],truncation=True,max_length=128)

m_en=sub_en.map(m_tok_fn,batched=True)
m_hi=hi.map(m_tok_fn,batched=True)
m_split=m_en.train_test_split(0.1)

m_mdl=AutoModelForSequenceClassification.from_pretrained("bert-base-multilingual-cased",num_labels=3)

m_tr=Trainer(
    model=m_mdl,
    args=args,
    train_dataset=m_split['train'],
    eval_dataset=m_split['test'],
    tokenizer=m_tok,
    compute_metrics=acc
)
m_tr.train()

m_p=m_tr.predict(m_hi)
print("mbert acc:", accuracy_score(m_p.label_ids, np.argmax(m_p.predictions,axis=-1)))

In [ ]:
#onnx stuff
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

om=ORTModelForSequenceClassification.from_pretrained("my_model",export=True,provider="CPUExecutionProvider",local_files_only=True)
om.save_pretrained("onnx")
tok.save_pretrained("onnx")

q=ORTQuantizer.from_pretrained("onnx",file_name="model.onnx")
conf=AutoQuantizationConfig.avx512_vnni(is_static=False,per_channel=False)
q.quantize(save_dir="onnx_q",quantization_config=conf)
print("done converting")